# Phy-L JEPA Colab

Notebook Colab pour lancer le masked image JEPA sur GPU, puis entrainer/tester les probes sain/cancer.

Remplace `<TON_URL_GITHUB>` par ton URL GitHub si le repo n est pas deja clone.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!test -d Phy-L-Jepa || git clone <TON_URL_GITHUB> Phy-L-Jepa
%cd /content/Phy-L-Jepa
!git pull

In [ ]:
!pip -q install numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_ROOT = '/content/drive/MyDrive/GIGADATASET_COLAB_NPZ'
IMAGE_OUT = '/content/Phy-L-Jepa/results/masked_image_jepa_gpu'
IMAGE_PROBE_OUT = '/content/Phy-L-Jepa/results/masked_image_probe_gpu'
IMAGE_CKPT = IMAGE_OUT + '/masked_image_jepa/latest.pth.tar'
print('DATA_ROOT =', DATA_ROOT)
print('IMAGE_CKPT =', IMAGE_CKPT)

## Train masked image JEPA

Mode image-scale: patch embedding sur les 16 canaux Mueller, transformer context encoder, AMP CUDA si GPU disponible.

In [ ]:
!python train_jepa_image_gpu.py \
  --data-root "$DATA_ROOT" \
  --epochs 150 \
  --batch-size 512 \
  --embed-dim 128 \
  --depth 4 \
  --num-heads 8 \
  --mlp-hidden-dim 256 \
  --mask-ratio 0.6 \
  --output-dir "$IMAGE_OUT"

## Train probes

JEPA gele, puis tete lineaire et tete MLP pour predire sain/cancer.

In [ ]:
!python train_probe_mlp.py \
  --encoder-type image \
  --data-root "$DATA_ROOT" \
  --jepa-ckpt "$IMAGE_CKPT" \
  --output-dir "$IMAGE_PROBE_OUT" \
  --epochs 40 \
  --batch-size 512 \
  --hidden-dim 64 \
  --dropout 0.25 \
  --patience 8

## Test only

Evaluation finale sur le split test/cohort1 sans relancer l entrainement des probes.

In [ ]:
!python eval_probe_mlp.py \
  --encoder-type image \
  --data-root "$DATA_ROOT" \
  --jepa-ckpt "$IMAGE_CKPT" \
  --suite-dir "$IMAGE_PROBE_OUT" \
  --kind both

## Ancien mode Cloude transformer

Garde ces cellules si tu veux comparer avec le mode matrice/Cloude.

In [ ]:
!python train_jepa_cpu_150.py --data-root "$DATA_ROOT" --epochs 150 --batch-size 256 --output-dir /content/Phy-L-Jepa/results/phys_jepa_cloude_transformer_gpu

In [ ]:
!python train_probe_mlp.py --encoder-type cloude --data-root "$DATA_ROOT" --jepa-ckpt /content/Phy-L-Jepa/results/phys_jepa_cloude_transformer_gpu/phys_jepa_cloude_transformer/latest.pth.tar --output-dir /content/Phy-L-Jepa/results/phys_jepa_probe_suite_full_gpu